In [ ]:
!pip install -q osmnx networkx pandas scikit-learn

In [ ]:
# ============================================================
# 1. DATOS: 30 puntos de interés de Gran Canaria
# ============================================================
import io, os
import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox

ox.settings.use_cache = True   # evita volver a descargar lo mismo en cada ejecución

csv_data = """Nombre,Latitud,Longitud,Puntuacion,Tiempo_Visita_Min
Roque Nublo,27.9619,-15.6015,10,90
Dunas de Maspalomas,27.7410,-15.5866,10,120
Puerto de Mogán,27.8173,-15.7645,9,90
Vegueta (Las Palmas),28.1011,-15.4143,9,150
Pico de las Nieves,27.9620,-15.5714,9,45
Teror (Basílica),28.0589,-15.5475,8,60
Arucas (Iglesia),28.1186,-15.5222,8,60
Barranco de Guayadeque,27.934100,-15.480720,7,90
Cenobio de Valerón,28.1362,-15.6033,7,45
Puerto de las Nieves (Agaete),28.1026,-15.7071,8,90
Playa de las Canteras,28.1408,-15.4349,10,120
Jardín Botánico Viera y Clavijo,28.0655,-15.4619,8,90
Cueva Pintada (Gáldar),28.1451,-15.6540,9,60
Mirador de Unamuno,28.0205,-15.6469,7,30
Tejeda (Pueblo),27.9946,-15.6152,9,60
Bufadero de la Garita,27.9926,-15.3853,7,30
Presa de las Niñas,27.928942,-15.665886,7,60
Playa del Inglés,27.7562,-15.5694,8,120
Caldera de Bandama,28.0312,-15.4566,8,60
Los Azulejos de Veneguera,27.921477,-15.725542,7,30
Cueva del Rey,27.9734,-15.6318,8,60
Firgas (Paseo de Gran Canaria),28.1086,-15.5638,7,45
Mirador del Balcón,28.019483,-15.785296,9,30
Barranco de los Cernícalos,27.9691,-15.4746,8,120
Charco de San Lorenzo,28.1444,-15.5898,7,60
Tufia (Poblado y Playa),27.9602,-15.3822,7,45
Jardín de la Marquesa (Arucas),28.1205,-15.5342,7,60
Mundo Aborigen,27.813069,-15.582363,7,90
Barranco de Fataga,27.860383,-15.568587,8,45
Playa de Amadores,27.7946,-15.7233,8,120"""

df = pd.read_csv(io.StringIO(csv_data))

# Validación: detecta coordenadas mal escritas (p. ej. 279.619 en vez de 27.9619)
assert df["Latitud"].between(27.6, 28.3).all(), "Hay latitudes fuera de Gran Canaria"
assert df["Longitud"].between(-15.9, -15.3).all(), "Hay longitudes fuera de Gran Canaria"
assert df["Nombre"].is_unique, "Hay nombres repetidos"
print(f"{len(df)} puntos cargados y validados.")

# ------------------------------------------------------------
# Alojamientos: base de salida y regreso de cada jornada.
# NO son puntos de interés: puntuación 0 y tiempo de visita 0.
# (nombre, latitud, longitud, carpeta de salida)
# ------------------------------------------------------------
HOTELES = [
    ("Hotel Santa Catalina (Las Palmas)", 28.121080, -15.428184, "santa_catalina"),
    ("Hotel Lopesan Villa del Conde", 27.742417, -15.606604, "lopesan_villa_del_conde"),
]
df_h = pd.DataFrame(HOTELES, columns=["Nombre", "Latitud", "Longitud", "carpeta"])
df_h["Puntuacion"] = 0
df_h["Tiempo_Visita_Min"] = 0
assert df_h["Latitud"].between(27.6, 28.3).all() and df_h["Longitud"].between(-15.9, -15.3).all()
assert not set(df_h["Nombre"]) & set(df["Nombre"]), "Un hotel repite el nombre de un POI"
print(f"{len(df_h)} hoteles definidos.")

In [ ]:
# ============================================================
# 2. GRAFO DE CARRETERAS (con caché en disco)
# ============================================================
GRAFO_PATH = "gran_canaria_drive.graphml"

if os.path.exists(GRAFO_PATH):
    G = ox.load_graphml(GRAFO_PATH)
    print("Grafo cargado desde disco.")
else:
    print("Descargando la red de carreteras (un par de minutos)...")
    G = ox.graph_from_place("Gran Canaria, Spain", network_type="drive")

    # CLAVE: quedarse solo con la mayor componente FUERTEMENTE conexa.
    # Así desde cualquier nodo se puede llegar a cualquier otro y ya no
    # aparecen 'inf' por callejones sin salida o tramos de sentido único aislados.
    G = ox.truncate.largest_component(G, strongly=True)

    G = ox.add_edge_speeds(G)        # velocidad según tipo de vía / maxspeed
    G = ox.add_edge_travel_times(G)  # segundos por tramo
    ox.save_graphml(G, GRAFO_PATH)

print(f"Grafo: {len(G.nodes):,} nodos, {len(G.edges):,} tramos")

In [ ]:
# ============================================================
# 3. EMPAREJAR CADA PUNTO CON SU NODO DE CARRETERA MÁS CERCANO
# ============================================================
nodos, dist_m = ox.distance.nearest_nodes(
    G, X=df["Longitud"].values, Y=df["Latitud"].values, return_dist=True
)
df["nodo"] = nodos
df["dist_snap_m"] = np.round(dist_m, 0)

# Un punto muy lejos de la carretera suele indicar una coordenada mal puesta
lejos = df[df["dist_snap_m"] > 300]
if len(lejos):
    print("AVISO: puntos a más de 300 m de la carretera más cercana (revisar coordenadas):")
    print(lejos[["Nombre", "dist_snap_m"]].to_string(index=False))

# Dos puntos distintos pegados al mismo nodo darían tiempo 0 entre ellos
dup = df[df["nodo"].duplicated(keep=False)]
if len(dup):
    print("AVISO: puntos que comparten nodo de carretera:")
    print(dup[["Nombre", "nodo"]].to_string(index=False))

# ---- Hoteles: mismo emparejamiento con la red de carreteras ----
nodos_h, dist_h = ox.distance.nearest_nodes(
    G, X=df_h["Longitud"].values, Y=df_h["Latitud"].values, return_dist=True
)
df_h["nodo"] = nodos_h
df_h["dist_snap_m"] = np.round(dist_h, 0)
print("Hoteles -> distancia al nodo de carretera más cercano (m):")
print(df_h[["Nombre", "dist_snap_m"]].to_string(index=False))
if (df_h["dist_snap_m"] > 300).any():
    print("AVISO: algún hotel está a más de 300 m de la carretera (revisar coordenadas).")

# POIs primero y hoteles después: la matriz se calcula una sola vez con todo
df_all = pd.concat([df, df_h], ignore_index=True)

In [ ]:
# ============================================================
# 4a. PARÁMETROS DE CALIBRACIÓN
# ============================================================
# Tiempos en coche de Google Maps (día laborable, media mañana). OSMnx calcula
# a la velocidad máxima de cada vía, sin tráfico ni curvas, así que sale optimista.
PARES_GOOGLE = [
    # Urbanas y cortas
    ("Vegueta (Las Palmas)", "Playa de las Canteras", 11),
    # Autopista GC-1 y costa sur
    ("Vegueta (Las Palmas)", "Dunas de Maspalomas", 38),
    ("Playa del Inglés", "Puerto de Mogán", 26),
    ("Vegueta (Las Palmas)", "Playa de Amadores", 46),
    # Norte y noroeste
    ("Vegueta (Las Palmas)", "Cueva Pintada (Gáldar)", 32),
    ("Arucas (Iglesia)", "Puerto de las Nieves (Agaete)", 27),
    # Montaña e interior
    ("Roque Nublo", "Puerto de Mogán", 81),
    ("Tejeda (Pueblo)", "Teror (Basílica)", 46),
    ("Mirador de Unamuno", "Puerto de Mogán", 106),
    ("Pico de las Nieves", "Vegueta (Las Palmas)", 52),
    ("Mirador del Balcón", "Puerto de las Nieves (Agaete)", 35),
    ("Barranco de Fataga", "Dunas de Maspalomas", 28),
]
PARES_GOOGLE = [p for p in PARES_GOOGLE if p[2] is not None]

# --- Penalización de vías sinuosas (celda 4b) ---
PENALIZAR_CURVAS = True
K_CANDIDATOS = [3000, 1500, 1000, 700, 500, 350]  # grados/km; menor K = más penalización
K_MANUAL = None        # pon un número para fijar K sin buscarlo
MEJORA_MIN = 0.5       # solo se activa si reduce el error medio al menos esto (min)

# --- Factor global final (celda 5b) ---
# "origen"  = mínimos cuadrados proporcional (más conservador con rutas largas)
# "mediana" = mediana de los ratios (más robusta a pares raros)
METODO_FACTOR = "origen"
FACTOR_MANUAL = None   # pon un número (p. ej. 1.25) para saltarte el cálculo

In [ ]:
# ============================================================
# 4b. PENALIZACIÓN DE VÍAS SINUOSAS
# ============================================================
# Idea: una carretera de montaña con muchas curvas no se recorre a su límite de
# velocidad. Para cada tramo medimos cuántos grados gira por kilómetro (curvatura)
# y reducimos su velocidad:
#       v_efectiva = v_limite / (1 + curvatura / K)
# K controla la fuerza (menor K = más penalización) y se elige comparando con Google.
VEL_MIN_KMH = 15.0   # suelo de velocidad
LONG_MIN_M = 50.0    # tramos más cortos no se penalizan (la curvatura no es fiable)

def curvatura_deg_km(G, u, v, data):
    largo = data.get("length", 0.0) or 0.0
    if largo < LONG_MIN_M:
        return 0.0
    geom = data.get("geometry")
    if geom is not None:
        pts = list(geom.coords)
    else:
        pts = [(G.nodes[u]["x"], G.nodes[u]["y"]), (G.nodes[v]["x"], G.nodes[v]["y"])]
    if len(pts) < 3:
        return 0.0
    lat0 = np.radians(np.mean([p[1] for p in pts]))
    xy = np.array([[(p[0] - pts[0][0]) * 111_320 * np.cos(lat0),
                    (p[1] - pts[0][1]) * 110_574] for p in pts])
    d = np.diff(xy, axis=0)
    d = d[np.hypot(d[:, 0], d[:, 1]) > 1.0]          # descarta vértices casi duplicados
    if len(d) < 2:
        return 0.0
    rumbo = np.arctan2(d[:, 1], d[:, 0])
    giro = np.abs((np.diff(rumbo) + np.pi) % (2 * np.pi) - np.pi)
    return min(float(np.degrees(giro).sum()) / (largo / 1000.0), 3000.0)

def aplicar_penalizacion(G, K):
    for u, v, k, data in G.edges(keys=True, data=True):
        vel = data["speed_kph"]
        if K is not None:
            vel = max(vel / (1 + data["curv_deg_km"] / K), min(VEL_MIN_KMH, vel))
        data["travel_time"] = data["length"] / (vel * 1000 / 3600)

for u, v, k, data in G.edges(keys=True, data=True):
    data["curv_deg_km"] = curvatura_deg_km(G, u, v, data)

curv = np.array([d["curv_deg_km"] for _, _, d in G.edges(data=True)])
print(f"Curvatura (grados/km) -> mediana: {np.median(curv):.0f} | "
      f"percentil 90: {np.percentile(curv, 90):.0f} | máx: {curv.max():.0f}")

nodo_de = dict(zip(df["Nombre"], df["nodo"]))

def tiempos_pares(G):
    por_origen = {}
    for o in {o for o, _, _ in PARES_GOOGLE}:
        por_origen[o] = nx.single_source_dijkstra_path_length(G, nodo_de[o], weight="travel_time")
    return np.array([por_origen[o][nodo_de[d]] / 60.0 for o, d, _ in PARES_GOOGLE])

K_ELEGIDO = None
if PENALIZAR_CURVAS and K_MANUAL is not None:
    K_ELEGIDO = K_MANUAL
    print(f"K fijado a mano: {K_ELEGIDO}")
elif PENALIZAR_CURVAS and PARES_GOOGLE:
    goo = np.array([g for _, _, g in PARES_GOOGLE], dtype=float)
    filas = []
    for K in [None] + K_CANDIDATOS:
        aplicar_penalizacion(G, K)
        m = tiempos_pares(G)
        f = float(m @ goo / (m @ m))             # factor global que quedaría después
        filas.append((K, f, float(np.mean(np.abs(m * f - goo)))))
    tabla_k = pd.DataFrame(filas, columns=["K", "factor_final", "error_medio_min"])
    tabla_k["K"] = tabla_k["K"].astype(object).where(tabla_k["K"].notna(), "sin penalizar")
    print(tabla_k.round(2).to_string(index=False))
    mejor = int(np.argmin(tabla_k["error_medio_min"].values))
    if tabla_k["error_medio_min"].iloc[0] - tabla_k["error_medio_min"].iloc[mejor] >= MEJORA_MIN:
        K_ELEGIDO = ([None] + K_CANDIDATOS)[mejor]
    print("K elegido:", K_ELEGIDO if K_ELEGIDO is not None else "ninguno (no mejora lo bastante)")

aplicar_penalizacion(G, K_ELEGIDO)

In [ ]:
# ============================================================
# 5. MATRIZ DE TIEMPOS N x N (minutos), POIs + hoteles
# ============================================================
# Un Dijkstra por origen (N ejecuciones) en lugar de uno por cada pareja (N^2).
n = len(df_all)
T = np.full((n, n), np.inf)

for i, origen in enumerate(df_all["nodo"]):
    tiempos = nx.single_source_dijkstra_path_length(G, origen, weight="travel_time")
    for j, destino in enumerate(df_all["nodo"]):
        if destino in tiempos:
            T[i, j] = tiempos[destino] / 60.0   # segundos -> minutos

T = np.round(T, 2)
np.fill_diagonal(T, 0.0)

matriz = pd.DataFrame(T, index=df_all["Nombre"], columns=df_all["Nombre"])

# ---- Comprobaciones antes de exportar ----
assert not np.isinf(T).any(), "Quedan 'inf' en la matriz"
assert not np.isnan(T).any(), "Hay NaN en la matriz"
off = T[~np.eye(n, dtype=bool)]
print(f"Tiempo mínimo entre puntos: {off.min():.2f} min | máximo: {off.max():.2f} min")
print(f"Asimetría máxima |T[i,j]-T[j,i]|: {np.abs(T - T.T).max():.2f} min "
      "(normal: hay calles de sentido único)")
matriz.iloc[:5, :5]

In [ ]:
# ============================================================
# 5b. FACTOR GLOBAL DE CALIBRACIÓN CON GOOGLE MAPS
# (se mide con pares de POIs y se aplica a toda la matriz, hoteles incluidos)
# ============================================================
if PARES_GOOGLE:
    modelo = np.array([matriz.loc[o, d] for o, d, _ in PARES_GOOGLE])
    goo = np.array([g for _, _, g in PARES_GOOGLE], dtype=float)
    ratios = goo / modelo

    if METODO_FACTOR == "mediana":
        factor = float(np.median(ratios))
    else:
        factor = float(modelo @ goo / (modelo @ modelo))
    if FACTOR_MANUAL is not None:
        factor = float(FACTOR_MANUAL)

    tabla = pd.DataFrame({
        "origen": [o for o, _, _ in PARES_GOOGLE],
        "destino": [d for _, d, _ in PARES_GOOGLE],
        "modelo_min": modelo, "google_min": goo, "ratio": ratios.round(2),
        "calibrado_min": (modelo * factor).round(1),
        "error_min": (modelo * factor - goo).round(1),   # negativo = sigue optimista
    })
    print(tabla.to_string(index=False))
    print()
    print(f"Factor de calibración ({'manual' if FACTOR_MANUAL is not None else METODO_FACTOR}): x{factor:.2f}")
    print(f"Error medio antes: {np.mean(np.abs(modelo - goo)):.1f} min | "
          f"después: {np.mean(np.abs(modelo * factor - goo)):.1f} min")
    if ratios.std() / ratios.mean() > 0.25:
        print("AVISO: los ratios varían mucho entre parejas; un único factor no basta.")

    matriz_osm = matriz.copy()                  # copia sin el factor global
    matriz = (matriz * factor).round(2)
else:
    print("Sin pares de Google Maps: la matriz queda SIN calibrar (tiempos optimistas).")

In [ ]:
# ============================================================
# 6. EXPORTACIÓN: un conjunto de datos por hotel (31 nodos cada uno)
# ============================================================
import zipfile

COLS = ["Nombre", "Latitud", "Longitud", "Puntuacion", "Tiempo_Visita_Min"]
POIS = df["Nombre"].tolist()
ARCHIVOS = ["matriz_tiempos.csv", "puntos_interes.csv", "coordenadas.csv"]

for nombre, _, _, carpeta in HOTELES:
    nombres = POIS + [nombre]                      # 30 POIs + este hotel
    os.makedirs(carpeta, exist_ok=True)
    matriz.loc[nombres, nombres].to_csv(f"{carpeta}/matriz_tiempos.csv", index_label="Nombre")
    puntos = pd.concat([df[COLS], df_h.loc[df_h["Nombre"] == nombre, COLS]], ignore_index=True)
    puntos.to_csv(f"{carpeta}/puntos_interes.csv", index=False)
    puntos[["Nombre", "Latitud", "Longitud"]].rename(
        columns={"Nombre": "nombre", "Latitud": "latitud", "Longitud": "longitud"}
    ).to_csv(f"{carpeta}/coordenadas.csv", index=False)

extra = ["Matriz_completa_con_hoteles.csv"]
matriz.to_csv(extra[0], index_label="Nombre")
if "matriz_osm" in globals():                      # copia antes del factor global
    matriz_osm.to_csv("Matriz_sin_factor_global.csv", index_label="Nombre")
    extra.append("Matriz_sin_factor_global.csv")

with zipfile.ZipFile("resultados.zip", "w") as z:
    for _, _, _, carpeta in HOTELES:
        for f in ARCHIVOS:
            z.write(f"{carpeta}/{f}")
    for f in extra:
        z.write(f)

print("\nResumen de tiempos desde/hasta cada hotel (minutos, ya calibrados):")
for nombre, _, _, carpeta in HOTELES:
    ida, vuelta = matriz.loc[nombre, POIS], matriz.loc[POIS, nombre]
    print(f"- {nombre}: ida {ida.min():.0f}-{ida.max():.0f} (más lejano: {ida.idxmax()}) | "
          f"vuelta {vuelta.min():.0f}-{vuelta.max():.0f}")
print("\nGuardado en resultados.zip ->", ", ".join(f"{c}/" for _, _, _, c in HOTELES), "+", ", ".join(extra))

try:  # descarga directa si estás en Colab
    from google.colab import files
    files.download("resultados.zip")
except ImportError:
    pass